# Hamiltonians with SparsePauliOp

Construct molecular and lattice Hamiltonians using Qiskit's `SparsePauliOp`, then evaluate expectation values on `Statevector` states.

In [ ]:
import numpy as np
import qiskit as qk

## H2 Hamiltonian (2-qubit representation)

Pauli coefficients for H2 at equilibrium bond length in a minimal 2-qubit encoding.

In [ ]:
H2_PAULI_SUM = [
    (-0.81261, "II"),
    (0.17120, "IZ"),
    (-0.22279, "ZI"),
    (0.17120, "ZZ"),
    (0.04532, "XX"),
]

def build_pauli_hamiltonian(pauli_terms):
    coeffs = [complex(c) for c, _ in pauli_terms]
    labels = [label for _, label in pauli_terms]
    return qk.quantum_info.SparsePauliOp.from_list(list(zip(labels, coeffs)))

H = build_pauli_hamiltonian(H2_PAULI_SUM)
print(f"SparsePauliOp: {H.num_qubits} qubits, {len(H)} terms")
for coeff, label in zip(H.coeffs, H.paulis.to_labels()):
    print(f"  {coeff.real:+.5f} · {label}")

## Matrix representation and exact eigenvalues

In [ ]:
mat = H.to_matrix()
eigenvalues = np.linalg.eigvalsh(mat)
print("Hamiltonian matrix:")
print(np.array2string(mat.real, precision=4, suppress_small=True))
print(f"\nEigenvalues: {np.round(eigenvalues, 6)}")
print(f"Ground state energy: {eigenvalues[0]:.6f}")

## Expectation values on example states

In [ ]:
sv_backend = qk.quantum_info.Statevector

def expectation_value(state, hamiltonian):
    return float(np.real(state.expectation_value(hamiltonian)))

# |00⟩
state_00 = sv_backend.from_int(0, dims=4)
print(f"<00|H|00> = {expectation_value(state_00, H):.6f}")

# |11⟩
state_11 = sv_backend.from_int(3, dims=4)
print(f"<11|H|11> = {expectation_value(state_11, H):.6f}")

# |++⟩
plus = sv_backend.from_label("++")
print(f"<++|H|++> = {expectation_value(plus, H):.6f}")

# Exact ground state
_, eigvecs = np.linalg.eigh(mat)
gs_vec = np.ascontiguousarray(eigvecs[:, 0])
ground = sv_backend(gs_vec)
print(f"<GS|H|GS> = {expectation_value(ground, H):.6f}  (exact ground state)")

## Custom ZZ + transverse field lattice Hamiltonian

In [ ]:
lattice_terms = [
    (1.0, "ZZ"),
    (0.5, "XI"),
    (0.5, "IX"),
]
H_lattice = build_pauli_hamiltonian(lattice_terms)
print(f"H = {H_lattice}")

mat_lattice = H_lattice.to_matrix()
evals = np.linalg.eigvalsh(mat_lattice)
print(f"Eigenvalues: {np.round(evals, 6)}")
print(f"Ground state energy: {evals[0]:.6f}")